In [1]:
import requests
from bs4 import BeautifulSoup
import json
import time
import os
from urllib.parse import urljoin


In [2]:
# Where to save the scraped articles
OUTPUT_DIR = "data/raw"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# How many articles to collect per category
MAX_ARTICLES_PER_CATEGORY = 100

# Polite delay between requests in seconds
DELAY = 1.5

# Always include this so PetMD doesn't reject us as a bot
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
}

In [3]:
CATEGORIES = [
    # Dogs
    {"url": "https://www.petmd.com/dog/allergies", "species": "dog", "topic": "allergies"},
    {"url": "https://www.petmd.com/dog/symptoms", "species": "dog", "topic": "symptoms"},
    {"url": "https://www.petmd.com/dog/care", "species": "dog", "topic": "care"},
    {"url": "https://www.petmd.com/dog/conditions", "species": "dog", "topic": "conditions"},
    {"url": "https://www.petmd.com/dog/centers/nutrition", "species": "dog", "topic": "food_safety"},
    {"url": "https://www.petmd.com/dog/emergency/poisoning-toxicity", "species": "dog", "topic": "poisoning"},
    {"url": "https://www.petmd.com/dog/behavior", "species": "dog", "topic": "behavior"},
    {"url": "https://www.petmd.com/dog/puppy", "species": "dog", "topic": "puppies"},
    {"url": "https://www.petmd.com/dog/senior", "species": "dog", "topic": "senior"},
    {"url": "https://www.petmd.com/dog/adult", "species": "dog", "topic": "general"},


    # Cats
    {"url": "https://www.petmd.com/cat/allergies", "species": "cat", "topic": "allergies"},
    {"url": "https://www.petmd.com/cat/symptoms", "species": "cat", "topic": "symptoms"},
    {"url": "https://www.petmd.com/cat/care", "species": "cat", "topic": "care"},
    {"url": "https://www.petmd.com/cat/conditions", "species": "cat", "topic": "conditions"},
    {"url": "https://www.petmd.com/cat/centers/nutrition", "species": "cat", "topic": "food_safety"},
    {"url": "https://www.petmd.com/cat/emergency/poisoning-toxicity", "species": "cat", "topic": "poisoning"},
    {"url": "https://www.petmd.com/cat/behavior", "species": "cat", "topic": "behavior"},
    {"url": "https://www.petmd.com/cat/kitten", "species": "cat", "topic": "kittens"},
    {"url": "https://www.petmd.com/cat/senior", "species": "cat", "topic": "senior"},
    {"url": "https://www.petmd.com/cat/adult", "species": "cat", "topic": "general"},

]

In [9]:
def get_article_links(category_url, species, topic):
    """Visit a category page and return up to MAX_ARTICLES_PER_CATEGORY article links."""
    try:
        response = requests.get(category_url, headers=HEADERS, timeout=10)
        response.raise_for_status()  # raises an error if status is 4xx or 5xx
        soup = BeautifulSoup(response.content, "html.parser")
        
        # Get all links on the page
        all_links = soup.find_all("a", href=True)
        
        # Keep only links that look like articles
        # Articles have longer paths with multiple segments
        # Navigation links are short like /dog/conditions
        article_links = []
        for link in all_links:
            href = link["href"]
            # Article URLs have at least 3 path segments
            if href.count("/") >= 3 and not href.startswith("http"):
                full_url = urljoin("https://www.petmd.com", href)
                if full_url not in article_links:
                    article_links.append(full_url)
        
        return article_links[:MAX_ARTICLES_PER_CATEGORY]
    
    except Exception as e:
        print(f"Failed to get links from {category_url}: {e}")
        return []  # return empty list so the scraper keeps going

In [10]:
def scrape_article(url, species, topic):
    """Visit a single article and extract its content."""
    try:
        response = requests.get(url, headers=HEADERS, timeout=10)
        response.raise_for_status()
        soup = BeautifulSoup(response.content, "html.parser")
        
        # Extract title
        title_tag = soup.find("h1")
        title = title_tag.get_text(strip=True) if title_tag else "Unknown Title"
        
        # Find article body using partial class match
        # The full class has a generated hash suffix so we match partially
        body = soup.find("div", class_=lambda c: c and "article_body" in c)
        
        if not body:
            return None  # skip this article if we can't find the content
        
        # Extract clean text from all paragraphs in the body
        paragraphs = body.find_all("p")
        content = " ".join(p.get_text(strip=True) for p in paragraphs)
        
        if not content:
            return None  # skip if content is empty
        
        return {
            "title": title,
            "content": content,
            "url": url,
            "source": "PetMD",
            "species": species,
            "topic": topic
        }
    
    except Exception as e:
        print(f"Failed to scrape {url}: {e}")
        return None

In [11]:
def run_scraper(categories):
    """Main scraper loop — runs through all categories and saves articles."""
    total_saved = 0
    
    for category in categories:
        url = category["url"]
        species = category["species"]
        topic = category["topic"]
        
        print(f"\nScraping category: {species} / {topic}")
        
        # Step 1: get article links from category page
        article_links = get_article_links(url, species, topic)
        print(f"Found {len(article_links)} article links")
        
        time.sleep(DELAY)
        
        # Step 2: scrape each article
        for article_url in article_links:
            # Save as JSON file
            # Use a filename based on the URL to avoid duplicates
            filename = article_url.replace("https://www.petmd.com/", "").replace("/", "_") + ".json"
            filepath = os.path.join(OUTPUT_DIR, filename)
            
            if os.path.exists(filepath):
                print(f"Skipping (already exists): {filename[:50]}")
                continue
            article = scrape_article(article_url, species, topic)
                
            if article:
                with open(filepath, "w", encoding="utf-8") as f:
                    json.dump(article, f, ensure_ascii=False, indent=2)
          
                total_saved += 1
                print(f"Saved: {article['title'][:50]}")
                
            time.sleep(DELAY)
        
    print(f"\nDone. Total articles saved: {total_saved}")

# Run it
run_scraper(CATEGORIES)


Scraping category: dog / allergies
Found 23 article links
Skipping (already exists): dog_general-health_food-allergies-vs-seasonal-alle
Skipping (already exists): dog_conditions_skin_can-dogs-be-allergic-grass.jso
Skipping (already exists): dog_conditions_systemic_pollen-allergies-dogs.json
Skipping (already exists): dog_symptoms_itchy-dog.json
Skipping (already exists): dog_general-health_dog-allergic-reaction.json
Skipping (already exists): dog_general-health_can-dogs-be-allergic-to-cats.js
Skipping (already exists): dog_general-health_dog-skin-allergies.json
Skipping (already exists): dog_conditions_skin_seasonal-allergies-dogs.json
Skipping (already exists): dog_general-health_can-you-be-allergic-dog-saliva.
Skipping (already exists): dog_general-health_hypoallergenic-dogs.json
Skipping (already exists): dog_care_can-i-give-my-dog-benadryl-and-if-so-how-
Skipping (already exists): dog_procedure_dog-allergy-tests.json

Scraping category: dog / symptoms
Found 30 article links
Skippi

In [12]:
#list of all filenames in the output directory
files = os.listdir("data/raw")

#initialize empty files
empty = 0

#if the content is empty or too short, count it as empty
for f in files:
    with open(f"data/raw/{f}") as fp:
        article = json.load(fp)
    if not article["content"] or len(article["content"]) < 100:
        empty += 1
        print(f"Empty or too short: {article['title'][:50]}")
#print summary stats
print(f"Total files: {len(files)}")
print(f"Empty or too short: {empty}")

Total files: 380
Empty or too short: 0


In [13]:
lengths = []
for f in files:
    with open(f"data/raw/{f}") as fp:
        article = json.load(fp)
    lengths.append(len(article["content"]))

print(f"Shortest: {min(lengths)} chars")
print(f"Longest: {max(lengths)} chars")
print(f"Average: {sum(lengths)//len(lengths)} chars")
print(f"Under 500 chars: {sum(1 for l in lengths if l < 500)}")
print(f"Under 1000 chars: {sum(1 for l in lengths if l < 1000)}")

Shortest: 636 chars
Longest: 36592 chars
Average: 6181 chars
Under 500 chars: 0
Under 1000 chars: 3


In [14]:
for f in files:
    with open(f"data/raw/{f}") as fp:
        article = json.load(fp)
    if len(article["content"]) < 1000:
        print(f"{article['title']} — {len(article['content'])} chars — {article['species']}/{article['topic']}")

Adhesions of the Eye’s Iris and Swelling of Eye in Cats — 809 chars — cat/conditions
Ameba Infection in Cats — 945 chars — cat/conditions
9 Best Probiotics for Dogs in 2025, Recommended By Vets — 636 chars — dog/senior


In [6]:
from collections import Counter

_dist_counter = Counter()
_missing = []

for _fname in os.listdir(OUTPUT_DIR):
    if not _fname.endswith(".json"):
        continue
    with open(os.path.join(OUTPUT_DIR, _fname)) as _fp:
        _a = json.load(_fp)
    _dist_counter[(_a.get("species", "MISSING"), _a.get("topic", "MISSING"))] += 1
    for _field in ["title", "content", "url", "source", "species", "topic"]:
        if not _a.get(_field):
            _missing.append((_fname, _field))

print(f"{'SPECIES':<14} {'TOPIC':<18} {'COUNT':>6}")
print("-" * 40)
for (_sp, _tp), _count in sorted(_dist_counter.items()):
    print(f"{_sp:<14} {_tp:<18} {_count:>6}")
print("-" * 40)
print(f"{'TOTAL':<33} {sum(_dist_counter.values()):>6}")

if _missing:
    print(f"\nWARNING: {len(_missing)} missing field(s)")
    for _fn, _fl in _missing[:10]:
        print(f"  {_fn} — missing '{_fl}'")
else:
    print("\nAll articles have required fields.")

SPECIES        TOPIC               COUNT
----------------------------------------
cat            allergies               7
cat            behavior               14
cat            care                    5
cat            conditions             89
cat            food_safety            10
cat            kittens                10
cat            senior                 14
cat            symptoms                9
dog            allergies               7
dog            behavior               11
dog            care                    6
dog            conditions             89
dog            food_safety             9
dog            poisoning              13
dog            puppies                14
dog            senior                 12
dog            symptoms               11
----------------------------------------
TOTAL                                330

All articles have required fields.


In [5]:
import os
import json

# Species and topics that are out of scope
_exclude_species = {"bird"}
_exclude_topics  = {"breeds", "general"}

_deleted = 0
_kept    = 0

for _fname in os.listdir(OUTPUT_DIR):
    if not _fname.endswith(".json"):
        continue

    _fpath = os.path.join(OUTPUT_DIR, _fname)

    with open(_fpath) as _fp:
        _a = json.load(_fp)

    if _a.get("species") in _exclude_species or _a.get("topic") in _exclude_topics:
        os.remove(_fpath)
        _deleted += 1
        print(f"Deleted: {_fname[:60]}")
    else:
        _kept += 1

print(f"\nDeleted: {_deleted} articles")
print(f"Kept:    {_kept} articles")

Deleted: bird_conditions_reproductive_c_bd_egg_binding.json
Deleted: bird_conditions_respiratory_c_bd_Avian_Influenza.json
Deleted: bird_conditions_skin_c_bd_Psittacine_beak_and_feather_diseas
Deleted: bird_general-health_backyard-chickens.json
Deleted: bird_poisoning_can-birds-eat-chocolate.json
Deleted: cat_breeds_abyssinian.json
Deleted: cat_breeds_american-curl.json
Deleted: cat_breeds_american-shorthair.json
Deleted: cat_breeds_balinese.json
Deleted: cat_breeds_bengal.json
Deleted: cat_breeds_birman.json
Deleted: cat_breeds_bombay.json
Deleted: cat_breeds_british-shorthair.json
Deleted: cat_breeds_burmese.json
Deleted: cat_breeds_chartreux.json
Deleted: cat_breeds_cornish-rex.json
Deleted: cat_breeds_c_ct_american_bobtail.json
Deleted: cat_breeds_c_ct_american_domestic.json
Deleted: cat_breeds_c_ct_american_wirehair.json
Deleted: cat_breeds_c_ct_burmilla_cat.json
Deleted: cat_breeds_c_ct_california_spangled_cat.json
Deleted: cat_breeds_c_ct_colorpoint_shorthair.json
Deleted: cat_b